# Baseline Evaluation — Qwen2.5-7B on 1,000 Test Examples

**Project:** LoRA Fine-Tuning for Text Summarization  
**Model:** Qwen2.5-7B (no fine-tuning)  
**Dataset:** CNN/DailyMail (version 3.0.0)  
**Purpose:** Establish a baseline by evaluating the raw, untrained Qwen2.5-7B model on 1,000 test examples. This gives us a reference point to measure how much LoRA fine-tuning actually improves performance.

---

### What is a Baseline?
A baseline is the performance of a model **before any fine-tuning**. We run the pre-trained Qwen model as-is on summarization tasks. Since it has never been specifically trained for summarization on this dataset, we expect modest scores — but this is intentional. It shows us where we started and how much LoRA improved things.

---

### Notebook Flow
1. Install libraries
2. Check GPU
3. Authenticate with HuggingFace
4. Load the CNN/DailyMail test dataset (1,000 examples)
5. Load the Qwen2.5-7B model and tokenizer
6. Test on a single example
7. Run baseline evaluation on 1,000 examples
8. Calculate BLEU and ROUGE scores
9. Save results

## Step 1 — Install Required Libraries
We install all the necessary Python libraries:
- `transformers` — HuggingFace library to load and run pre-trained models like Qwen
- `datasets` — HuggingFace library to load benchmark datasets like CNN/DailyMail
- `evaluate` — HuggingFace library for computing BLEU and ROUGE scores
- `rouge-score` — backend used by the evaluate library for ROUGE computation
- `nltk` — used internally for tokenization during BLEU calculation
- `accelerate` — helps run models efficiently on GPU

In [ ]:
!pip install transformers datasets evaluate rouge-score nltk accelerate -q
print("All libraries installed successfully!")

## Step 2 — Check GPU Availability
We verify that a GPU is available and accessible via CUDA.

**What is CUDA?**  
CUDA is NVIDIA's platform that allows Python code to run on the GPU instead of CPU. Running a large language model like Qwen2.5-7B on CPU would be extremely slow — on a GPU it is much faster. PyTorch uses CUDA automatically when a GPU is available.

**Note on GPU memory:**  
Qwen2.5-7B is a 7-billion parameter model. It requires a GPU with sufficient VRAM to load. The more VRAM available, the larger the batch size you can use during evaluation.

In [ ]:
import torch

# Check if GPU is available
if torch.cuda.is_available():
    print(f"   GPU is available!")
    print(f"   GPU Name     : {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   No GPU found. Please enable a GPU in your runtime settings.")
    print("   Qwen2.5-7B requires a GPU to run in reasonable time.")

## Step 3 — Authenticate with HuggingFace
Qwen2.5-7B is a publicly available model — unlike Llama, **no license acceptance is required**. You simply need a HuggingFace account and access token to download it.

1. Create a HuggingFace account at: https://huggingface.co/join
2. Generate an access token at: https://huggingface.co/settings/tokens

Paste your token when prompted below.

In [ ]:
from huggingface_hub import login

# This will prompt you to enter your HuggingFace access token
# Get your token from: https://huggingface.co/settings/tokens
# Note: Qwen2.5-7B is publicly available — no license acceptance needed
login()
print("Logged in to HuggingFace!")

## Step 4 — Load the CNN/DailyMail Test Dataset (1,000 examples)
We load only 1,000 examples from the CNN/DailyMail test split for this notebook.

**Why CNN/DailyMail?**  
It is the industry-standard benchmark dataset for news summarization research. It contains ~312k article-summary pairs from CNN and Daily Mail news articles. Using a public, well-known dataset makes our results reproducible and comparable with other research.

**Why only 1,000 examples here?**  
This is a quick proof-of-concept evaluation. For the full 11,490-example baseline, see `baseline_eval_11k.ipynb`.

In [ ]:
from datasets import load_dataset

print("Loading CNN/DailyMail test set (1,000 examples)...")

# Load the test split and select first 1,000 examples
# Version 3.0.0 is the standard version used in research
test_dataset = load_dataset("cnn_dailymail", "3.0.0", split="test")
test_dataset_1k = test_dataset.select(range(1000))

print(f"   Loaded {len(test_dataset_1k)} test examples")
print(f"\nDataset columns: {test_dataset_1k.column_names}")
print(f"\n--- Sample Example ---")
print(f"Article (first 300 chars) : {test_dataset_1k[0]['article'][:300]}")
print(f"\nReference Summary         : {test_dataset_1k[0]['highlights']}")

## Step 5 — Load the Qwen2.5-7B Model and Tokenizer

**What is a Tokenizer?**  
A tokenizer converts raw text into numbers (tokens) that the model understands. It also does the reverse — converting model output numbers back into readable text.

**What is AutoModelForCausalLM?**  
This loads a causal language model — a model that generates text by predicting the next token based on all previous tokens. Qwen2.5 is a causal LM. "Auto" means HuggingFace automatically picks the right model architecture based on the model name.

**Why `torch_dtype=torch.bfloat16`?**  
This loads the model in bfloat16 (Brain Floating Point) format instead of 32-bit. It reduces memory usage significantly and is the recommended precision for Qwen2.5 models. Unlike float16, bfloat16 maintains the same dynamic range as float32, making training and inference more numerically stable.

**Why `device_map="auto"`?**  
This tells HuggingFace to automatically distribute the model across all available GPUs (or onto CPU if no GPU is found). It handles memory allocation without requiring manual device assignment.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen2.5-7B"

print(f"Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Qwen2.5 has a built-in padding token — no manual assignment needed
# Left-padding is required for batched generation with causal language models
tokenizer.padding_side = "left"

print(f"Loading model {MODEL_NAME}...")
print("This may take a few minutes on first run (downloading ~15GB)...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,  # bfloat16 is recommended for Qwen2.5 — stable and memory-efficient
    device_map="auto"            # automatically distribute model across available GPU(s)
)

# Set model to evaluation mode — disables dropout layers used only during training
model.eval()

print(f"\n   Model loaded successfully!")
print(f"   Model device : {next(model.parameters()).device}")
print(f"   Parameters   : {sum(p.numel() for p in model.parameters()):,}")

## Step 6 — Test on a Single Example
Before running the full evaluation, we test on one example to make sure the model loads correctly and generates output.

**How does text generation work here?**
1. We build a prompt: `"Summarize the following article:\n\n{article}\n\nSummary:"`
2. The tokenizer converts this prompt into token IDs
3. The model generates new tokens after `"Summary:"`
4. The tokenizer decodes the output tokens back into text
5. We extract only the part after `"Summary:"` as the generated summary

**What is `torch.no_grad()`?**  
During inference (not training), we don't need to compute gradients. `torch.no_grad()` disables gradient tracking, saving memory and speeding up inference significantly.

In [ ]:
import time

# Pick the first test example
test_example = test_dataset_1k[0]

# Build the prompt — this tells the model what task to perform
# The model will generate text that continues after "Summary:"
prompt = f"Summarize the following article:\n\n{test_example['article']}\n\nSummary:"

# Tokenize the prompt
# truncation=True   : if article is too long, cut it to max_length
# max_length=512    : maximum number of input tokens
# return_tensors='pt': return PyTorch tensors (not plain lists)
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

# Move inputs to the same device as the model (GPU)
# Model and data must be on the same device or PyTorch will throw an error
inputs = {k: v.to(model.device) for k, v in inputs.items()}

start_time = time.time()

# Generate the summary
# torch.no_grad() disables gradient computation — we are only doing inference, not training
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,                    # generate at most 100 new tokens
        do_sample=False,                       # greedy decoding — always pick the most probable token
        pad_token_id=tokenizer.pad_token_id    # Qwen2.5 has its own pad token
    )

elapsed = time.time() - start_time

# Decode the output token IDs back into readable text
# skip_special_tokens=True removes tokens like <eos>, <pad> from the output
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract only the part after "Summary:" — the full output includes the input prompt
generated_summary = generated_text.split("Summary:")[-1].strip()[:200]

print("=" * 60)
print("SINGLE EXAMPLE TEST")
print("=" * 60)
print(f"Article (first 200 chars) : {test_example['article'][:200]}")
print(f"\nReference Summary         : {test_example['highlights']}")
print(f"\nGenerated Summary         : {generated_summary}")
print(f"\nGeneration Time           : {elapsed:.2f} seconds")
print("\n   Single example test passed! Proceeding to full evaluation...")

## Step 7 — Run Baseline Evaluation on 1,000 Examples
We now run the untrained Qwen model on all 1,000 test examples and collect generated summaries.

**Why sequential evaluation here?**  
For 1,000 examples, sequential (one-by-one) evaluation is straightforward and avoids memory issues. For larger evaluations (11k examples), we use batched evaluation which is significantly faster — see `baseline_eval_11k.ipynb`.

**What are we collecting?**
- `all_summaries` — the model's generated summaries
- `all_references` — the ground truth summaries from the dataset

We compare these two lists using BLEU and ROUGE metrics in the next step.

In [ ]:
from tqdm import tqdm
import time

all_summaries = []    # Will store model-generated summaries
all_references = []   # Will store ground truth reference summaries

print("Starting baseline evaluation on 1,000 test examples...")
print("=" * 60)

start_total = time.time()

# Loop through all 1,000 test examples
for i, example in enumerate(tqdm(test_dataset_1k, desc="Evaluating")):

    # Build the same prompt format used consistently across all experiments
    prompt = f"Summarize the following article:\n\n{example['article']}\n\nSummary:"

    # Tokenize and move to GPU
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Generate summary — no gradients needed during evaluation
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    # Decode and extract the summary portion
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_summary = generated_text.split("Summary:")[-1].strip()

    all_summaries.append(generated_summary)
    all_references.append(example["highlights"])

    # Print a progress update every 100 examples
    if (i + 1) % 100 == 0:
        elapsed = time.time() - start_total
        print(f"   [{i+1}/1000] completed | Time elapsed: {elapsed/60:.1f} mins")

total_time = time.time() - start_total
print(f"\n   Evaluation complete!")
print(f"   Total examples evaluated : {len(all_summaries)}")
print(f"   Total time               : {total_time/60:.1f} minutes")

## Step 8 — Calculate BLEU and ROUGE Scores

We now compare the model's generated summaries against the reference summaries using four metrics:

| Metric | What it measures |
|--------|------------------|
| **BLEU-4** | Overlap of 4-word phrases between generated and reference summary |
| **ROUGE-1** | Overlap of individual words (unigrams) |
| **ROUGE-2** | Overlap of 2-word phrases (bigrams) |
| **ROUGE-L** | Longest common subsequence between generated and reference |

**Note:** For abstractive summarization, scores in the range 10–40 are typical. The model paraphrases rather than copying — so low scores don't mean failure, they are expected. What matters is the **improvement** after fine-tuning.

In [ ]:
from evaluate import load
import numpy as np

print("Calculating BLEU and ROUGE scores...")
print("=" * 60)

# Load evaluation metrics from HuggingFace evaluate library
bleu_metric = load("bleu")
rouge_metric = load("rouge")

# Both BLEU and ROUGE receive raw strings — the evaluate library handles tokenization internally
# predictions : list of generated summary strings
# references  : for BLEU, each reference is wrapped in a list (supports multiple references per example)
#               for ROUGE, plain list of strings is fine
bleu_result = bleu_metric.compute(
    predictions=all_summaries,
    references=[[r] for r in all_references]   # each reference wrapped in a list
)

rouge_result = rouge_metric.compute(
    predictions=all_summaries,
    references=all_references                   # plain list of strings
)

# Extract BLEU-4 score (index 3 = 4-gram precision) and scale to 0-100
bleu4  = bleu_result['precisions'][3] * 100
rouge1 = rouge_result['rouge1'] * 100
rouge2 = rouge_result['rouge2'] * 100
rougeL = rouge_result['rougeL'] * 100

print(f"\n{'='*60}")
print(f"  BASELINE RESULTS — Qwen2.5-7B | 1,000 Test Examples")
print(f"{'='*60}")
print(f"  BLEU-4   : {bleu4:.2f}")
print(f"  ROUGE-1  : {rouge1:.2f}")
print(f"  ROUGE-2  : {rouge2:.2f}")
print(f"  ROUGE-L  : {rougeL:.2f}")
print(f"{'='*60}")
print(f"\nNote: These are baseline scores with NO fine-tuning.")
print(f"Compare with LoRA fine-tuned results in the lora_1k notebooks.")

## Step 9 — Save Results
We save the scores to a JSON file for easy reference and comparison across notebooks.

In [ ]:
import json

# Store all results in a dictionary
baseline_results_1k = {
    "experiment"  : "Baseline — Qwen2.5-7B (no fine-tuning)",
    "model"       : "Qwen/Qwen2.5-7B",
    "test_samples": 1000,
    "bleu_4"      : round(bleu4, 4),
    "rouge_1"     : round(rouge1, 4),
    "rouge_2"     : round(rouge2, 4),
    "rouge_L"     : round(rougeL, 4)
}

# Save to JSON file
with open("baseline_results_1k.json", "w") as f:
    json.dump(baseline_results_1k, f, indent=2)

print("   Results saved to baseline_results_1k.json")
print(json.dumps(baseline_results_1k, indent=2))